# 🎙️ Piper Offline Voice Cloning — Master Production Pipeline (Epoch 0 ➔ 100)
**High-Fidelity AI Voice Narration for Android & Moon+ Reader Pro (Zero PC Server, 100% Offline)**

### Overview:
* Fine-tune any voice from 1–3 hours of clean audio using a free **Google Colab T4 GPU**.
* Export an ultra-compact ~60 MB `.onnx` neural model + `tokens.txt`.
* Runs locally on Android phones via **SherpaTTS** (Next-gen Kaldi) with **RTF ~0.18** (5x faster than real-time, <5% battery/hr).

### Instructions:
1. Ensure Colab Runtime is set to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`).
2. Upload your clean narrator audio (e.g. `callum.m4a` or `callum.wav`) to the root of your **Google Drive**.
3. Execute the cells below in order.

## Cell 1: Environment Setup & Dependencies
Creates an isolated Python 3.10 virtual environment with `uv`, compiles the C++ `monotonic_align` engine, and downloads the English base checkpoint.

In [ ]:
from google.colab import drive
import os, glob, urllib.request

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. System dependencies
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg build-essential

# 3. Setup isolated Python 3.10 environment via uv
!pip install -q uv openai-whisper pydub
!uv python install 3.10
!rm -rf /content/piper_env
!uv venv --seed /content/piper_env --python 3.10

# 4. Install pinned dependencies (Prevents NumPy 2.0 & PyTorch Lightning deprecation crashes)
!uv pip install --python /content/piper_env torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!uv pip install --python /content/piper_env "numpy<2" "setuptools<70" pytorch-lightning==1.9.5 torchmetrics==0.11.4 piper-phonemize==1.1.0 onnx onnxruntime librosa cython tensorboard

# 5. Clone Piper and compile C++ monotonic alignment extension
!rm -rf /content/piper
!git clone -q https://github.com/rhasspy/piper.git /content/piper
!cd /content/piper/src/python && /content/piper_env/bin/python piper_train/vits/monotonic_align/setup.py build_ext --inplace
!mkdir -p /content/piper/src/python/piper_train/vits/monotonic_align/monotonic_align
!cp /content/piper/src/python/piper_train/vits/monotonic_align/core*.so /content/piper/src/python/piper_train/vits/monotonic_align/monotonic_align/ 2>/dev/null || true
!uv pip install --python /content/piper_env --no-deps -e /content/piper/src/python

# 6. Download official English medium base checkpoint (Ryan / Lessac)
os.makedirs('/content/base_model', exist_ok=True)
base_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/lessac/medium/epoch%3D2164-step%3D1355540.ckpt'
urllib.request.urlretrieve(base_url, '/content/base_model/base.ckpt')

print('✅ Setup complete! Base model and C++ extensions compiled.')


## Cell 2: Auto-Slice & Transcribe with Whisper
Finds your audio in Google Drive, segments it into 1.5s–10s chunks, and converts speech into phoneme alignment tensors.

In [ ]:
import os, glob, whisper
from pydub import AudioSegment

VOICE_SEARCH = 'callum' # Set your audio filename (without extension)

candidates = glob.glob(f'/content/drive/MyDrive/**/{VOICE_SEARCH}.*', recursive=True)
if not candidates:
    candidates = glob.glob('/content/drive/MyDrive/**/my_voice.*', recursive=True)
if not candidates:
    raise FileNotFoundError('❌ Could not find audio file in Google Drive! Please upload your audio.')

audio_path = candidates[0]
print(f'Found audio: {audio_path}')

# Transcribe and slice
print('⏳ Transcribing audio and cutting sentence slices (~3-5 mins)...')
asr = whisper.load_model('base.en')
result = asr.transcribe(audio_path, language='en')

sound = AudioSegment.from_file(audio_path).set_frame_rate(22050).set_channels(1).set_sample_width(2)
os.makedirs('/content/dataset/wav', exist_ok=True)

count = 0
with open('/content/dataset/metadata.csv', 'w', encoding='utf-8') as f:
    for seg in result['segments']:
        start_ms, end_ms = int(seg['start'] * 1000), int(seg['end'] * 1000)
        duration = end_ms - start_ms
        text = seg['text'].strip()
        if 1500 <= duration <= 10000 and len(text) > 3:
            chunk = sound[start_ms:end_ms]
            file_id = f'{count:05d}'
            chunk.export(f'/content/dataset/wav/{file_id}.wav', format='wav')
            f.write(f'{file_id}|{text}\n')
            count += 1

print(f'✅ Sliced {count} clean sentences.')

# Preprocess into phonemes
print('⏳ Converting dataset to phonemes...')
!PYTHONPATH=/content/piper/src/python /content/piper_env/bin/python3 -m piper_train.preprocess \
  --language en-us \
  --input-dir /content/dataset \
  --output-dir /content/dataset \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050

!ln -sf /content/dataset /content/preprocessed
print('✅ Preprocessing complete!')


## Cell 3: OOM-Proof GPU Training (Epoch 0 ➔ 70)
Trains the voice model on the GPU. Uses `batch-size 2` and `max-phoneme-ids 300` to guarantee zero CUDA Out-Of-Memory crashes on free 15GB T4 GPUs.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

CHECKPOINT_DIR = '/content/drive/MyDrive/Callum_3Hour_Master_Checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

!PYTHONPATH=/content/piper/src/python /content/piper_env/bin/python3 -m piper_train \
    --dataset-dir /content/dataset \
    --resume_from_single_speaker_checkpoint /content/base_model/base.ckpt \
    --checkpoint-epochs 10 \
    --max_epochs 70 \
    --batch-size 2 \
    --max-phoneme-ids 300 \
    --quality medium \
    --num-test-examples 0 \
    --default_root_dir "$CHECKPOINT_DIR"


## Cell 4: Resuming Across Accounts / Sessions (Epoch 70 ➔ 100)
If your Colab GPU time runs out, open a new session (or switch to Google Account 2), run **Cell 1**, and execute this cell to resume training to Epoch 100.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

RESUME_CKPT = '/content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/callum-master-epoch=69.ckpt'
CHECKPOINT_DIR = '/content/drive/MyDrive/Callum_3Hour_Master_Checkpoints'

!PYTHONPATH=/content/piper/src/python /content/piper_env/bin/python3 -m piper_train \
    --dataset-dir /content/dataset \
    --resume_from_checkpoint "$RESUME_CKPT" \
    --checkpoint-epochs 10 \
    --max_epochs 100 \
    --batch-size 2 \
    --max-phoneme-ids 300 \
    --quality medium \
    --num-test-examples 0 \
    --default_root_dir "$CHECKPOINT_DIR"


## Cell 5: Free Up Google Drive Quota
Checkpoints are ~806.7 MB each! Run this cell to delete early obsolete checkpoints (Epochs 09–49) and instantly reclaim ~4 GB of Google Drive space.

In [ ]:
# Reclaim ~4 GB by pruning redundant early checkpoints
!rm -f /content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/callum-master-epoch=09.ckpt
!rm -f /content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/callum-master-epoch=19.ckpt
!rm -f /content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/callum-master-epoch=29.ckpt
!rm -f /content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/callum-master-epoch=39.ckpt
!rm -f /content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/callum-master-epoch=49.ckpt
print('✅ Cleaned up early checkpoints. Drive space freed!')


## Cell 6: Fail-Safe ONNX Export, Tokens Generator & Browser Download
Exports the model to `.onnx`, generates `tokens.txt` directly from `phoneme_id_map`, copies `callum.onnx.json`, and triggers automatic browser downloads for all 3 files.

In [ ]:
# 1. Standalone Python exporter script (Bypasses Colab subshell quoting bugs)
export_code = '''
import sys
from pathlib import Path
import torch

sys.path.insert(0, "/content/piper/src/python")
from piper_train.vits.lightning import VitsModel

ckpt_path = Path("/content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/lightning_logs/version_3/checkpoints/epoch=99-step=74820.ckpt")
if not ckpt_path.is_file():
    import glob
    matches = glob.glob("/content/drive/MyDrive/Callum_3Hour_Master_Checkpoints/**/epoch=99*.ckpt", recursive=True)
    if matches:
        ckpt_path = Path(matches[-1])

output_path = Path("/content/drive/MyDrive/Callum_Epoch100_Final/callum.onnx")
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Loading checkpoint: {ckpt_path} ...")
assert ckpt_path.is_file(), f"Checkpoint not found at {ckpt_path}"

model = VitsModel.load_from_checkpoint(ckpt_path, dataset=None)
model_g = model.model_g
num_symbols = model_g.n_vocab
num_speakers = model_g.n_speakers

model_g.eval()
with torch.no_grad():
    model_g.dec.remove_weight_norm()

def infer_forward(text, text_lengths, scales, sid=None):
    noise_scale = scales[0]
    length_scale = scales[1]
    noise_scale_w = scales[2]
    audio = model_g.infer(
        text,
        text_lengths,
        noise_scale=noise_scale,
        length_scale=length_scale,
        noise_scale_w=noise_scale_w,
        sid=sid,
    )[0].unsqueeze(1)
    return audio

model_g.forward = infer_forward

dummy_input_length = 50
sequences = torch.randint(low=0, high=num_symbols, size=(1, dummy_input_length), dtype=torch.long)
sequence_lengths = torch.LongTensor([sequences.size(1)])
sid = torch.LongTensor([0]) if num_speakers > 1 else None
scales = torch.FloatTensor([0.667, 1.0, 0.8])
dummy_input = (sequences, sequence_lengths, scales, sid)

print("Exporting ONNX computational graph...")
torch.onnx.export(
    model=model_g,
    args=dummy_input,
    f=str(output_path),
    verbose=False,
    opset_version=15,
    input_names=["input", "input_lengths", "scales", "sid"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size", 1: "phonemes"},
        "input_lengths": {0: "batch_size"},
        "output": {0: "batch_size", 1: "time"},
    },
)
print(f"✅ Successfully created: {output_path} ({output_path.stat().st_size / (1024*1024):.2f} MB)")
'''

with open('/content/do_export.py', 'w') as f:
    f.write(export_code)

!/content/piper_env/bin/python3 /content/do_export.py

# 2. Generate tokens.txt from config
import json, os
from google.colab import files

final_dir = '/content/drive/MyDrive/Callum_Epoch100_Final'
config_path = '/content/dataset/config.json' if os.path.exists('/content/dataset/config.json') else '/content/preprocessed/config.json'

with open(config_path, 'r', encoding='utf-8') as f:
    cfg = json.load(f)

tokens_file = f'{final_dir}/tokens.txt'
with open(tokens_file, 'w', encoding='utf-8') as f:
    for s, i in cfg['phoneme_id_map'].items():
        if s == '\n': continue
        f.write(f"{s} {i[0] if isinstance(i, list) else i}\n")

!cp "$config_path" "$final_dir/callum.onnx.json"
os.sync()

print('\n--- FILES READY IN GOOGLE DRIVE ---')
!ls -lh "$final_dir"

print('\n--- TRIGGERING BROWSER DOWNLOADS ---')
files.download(f'{final_dir}/callum.onnx')
files.download(f'{final_dir}/tokens.txt')
files.download(f'{final_dir}/callum.onnx.json')
